# Data Prep
The purpose of this notebook is to transform the data in preparation for the algos. This involves scaling and transforming the numerical data. In addition there will be feature engineering.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import sys
import joblib

In [2]:
# set up relative imports
project_folder = Path.cwd().parent.parent
sys.path.append(str(project_folder))

In [3]:
from modeling.utilities.data_prep import f_to_c, convert_to_cyclical

In [4]:
df = pd.read_parquet('../data/clean-merged-data.parquet')

In [5]:
df.head(1)

,date,TMAX,TMIN,PRCP,BOT_TEMP_C,SURF_TEMP_C,month,day,year
0,1939-07-01,76.0,63.0,0.0,19.1,19.4,7,1,1939


In [6]:
# convert all temps to celcius
df['tmax_c'] = f_to_c(df.TMAX)
df['tmin_c'] = f_to_c(df.TMIN)

# add the sin and cos cyclical columns for day and month
df['sin_month'] = convert_to_cyclical(np.sin, df.month, 12)
df['cos_month'] = convert_to_cyclical(np.cos, df.month, 12)

# for day we are just assuming every month to 31 
df['sin_day'] = convert_to_cyclical(np.sin, df.day, 31)
df['cos_day'] = convert_to_cyclical(np.cos, df.day, 31)


In [7]:
df['past_10_day_ave_temp'] = df.SURF_TEMP_C.rolling(window=10,min_periods=1).mean()
df['past_10_day_std_temp'] = df.SURF_TEMP_C.rolling(window=10,min_periods=1).std()

In [8]:
# create the target variable
df['target'] = df.SURF_TEMP_C.shift(1)

In [9]:
df.head(5)

,date,TMAX,TMIN,PRCP,BOT_TEMP_C,SURF_TEMP_C,month,day,year,tmax_c,tmin_c,sin_month,cos_month,sin_day,cos_day,past_10_day_ave_temp,past_10_day_std_temp,target
0,1939-07-01,76.0,63.0,0.0,19.1,19.4,7,1,1939,24.444444,17.222222,-0.5,-0.866025,0.201299,0.979530,19.400000,NaN,NaN
1,1939-07-02,74.0,65.0,0.0,19.5,19.8,7,2,1939,23.333333,18.333333,-0.5,-0.866025,0.394356,0.918958,19.600000,0.282843,19.4
2,1939-07-03,71.0,62.0,0.0,19.5,20.1,7,3,1939,21.666667,16.666667,-0.5,-0.866025,0.571268,0.820763,19.766667,0.351188,19.8
3,1939-07-04,71.0,63.0,0.0,19.3,20.2,7,4,1939,21.666667,17.222222,-0.5,-0.866025,0.724793,0.688967,19.875000,0.359398,20.1
4,1939-07-05,72.0,64.0,0.0,17.6,20.6,7,5,1939,22.222222,17.777778,-0.5,-0.866025,0.848644,0.528964,20.020000,0.449444,20.2


In [10]:
df.shape

(31596, 18)

In [11]:
df.isnull().sum()

date                    0
TMAX                    0
TMIN                    0
PRCP                    0
BOT_TEMP_C              0
SURF_TEMP_C             0
month                   0
day                     0
year                    0
tmax_c                  0
tmin_c                  0
sin_month               0
cos_month               0
sin_day                 0
cos_day                 0
past_10_day_ave_temp    0
past_10_day_std_temp    1
target                  1
dtype: int64

In [12]:
df.dropna(inplace=True)
df.shape

(31595, 18)

#### split, scale, and save the data

In [13]:
# drop the date column
df.drop(columns=['date'], inplace=True)

In [14]:
train_dfs = []
test_dfs = []
val_dfs = []
window = 255

for start in range(0, len(df), window):
    end = min(start + window, len(df))

    slice_df = df[start: end]
    
    n = len(slice_df)
    train_end = int(.7 * n)
    val_end = int(.85*n)

    train_dfs.append(slice_df[:train_end])
    val_dfs.append(slice_df[train_end:val_end])
    test_dfs.append(slice_df[val_end:])


In [15]:
train_df = pd.concat(train_dfs)
test_df = pd.concat(test_dfs)
val_df = pd.concat(val_dfs)

In [16]:
target_col = 'target'
X_cols = list(df.columns)
X_cols.remove(target_col)

In [17]:
X_train = train_df[X_cols]
y_train = train_df[[target_col]]

In [18]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [19]:
X_train_scaled_df = pd.DataFrame(data=X_train_scaled, columns=X_cols)
X_train_scaled_df.to_parquet('../data/model-ready/water-weather/scaled-X-train.parquet')
y_train.to_parquet('../data/model-ready/water-weather/scaled-Y-train.parquet')

In [20]:
X_test = test_df[X_cols]
y_test = test_df[[target_col]]

X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(data=X_test_scaled, columns=X_cols)
X_test_scaled_df.to_parquet('../data/model-ready/water-weather/scaled-X-test.parquet')
y_test.to_parquet('../data/model-ready/water-weather/scaled-Y-test.parquet')

In [21]:
X_val = val_df[X_cols]
y_val = val_df[[target_col]]

X_val_scaled = scaler.transform(X_val)
X_val_scaled_df = pd.DataFrame(data=X_val_scaled, columns=X_cols)
X_val_scaled_df.to_parquet('../data/model-ready/water-weather/scaled-X-val.parquet')
y_val.to_parquet('../data/model-ready/water-weather/scaled-Y-val.parquet')

In [22]:
joblib.dump(scaler, '../data/model-ready/water-weather/minmax_scaler.joblib')

['../data/model-ready/water-weather/minmax_scaler.joblib']